# 유사 투수 매칭 (Similar Pitcher Finder)

## 목적
투수 이름/연도를 입력하면, 다음 조건을 만족하는 다른 투수-시즌 중 
릴리스 위치·익스텐션·팔각도 기준으로 가장 비슷한 투수-시즌 pair를 찾는다.

## 필터링 조건
1. 같은 GMM 클러스터 소속 ( `data_description.json` 기준)
2. 평균 구속이 비슷함 (기본값: ±2mph)
3. FIP(투수 평가지표)가 더 낮음 — *FIP 데이터는 아직 팀에서 미제공, 추후 추가 예정*

## 유사도 기준
위 조건을 통과한 후보들 중, 표준화한 (release_pos_x, release_pos_z, release_extension, arm_angle) 
4개 변수 기준 유클리드 거리가 가장 짧은 투수-시즌을 반환한다.

## 사용 데이터
- `data/processed/data_description.json`: 투수-시즌별 평균구속·클러스터 번호 (팀 제공)
- `data/raw/statcast_*.csv`: 투수-시즌별 릴리스 위치·익스텐션·팔각도 원본 (본인 계산)

## Step 1. Statcast 원본 데이터 불러오기

클러스터링 노트북(`sua_clustering.ipynb`)에서 이미 만들어둔 `data/raw`의 statcast csv들을
다시 불러온다. FF/SI/FC 필터링은 여기선 하지 않는다 — 팔각도·릴리스 위치는
투수의 모든 구종에 걸쳐 나타나는 "그 투수의 투구 폼 특성"이라 구종 상관없이 평균낸다.

In [1]:
import pandas as pd
import glob

files = glob.glob("../data/raw/statcast_*.csv")
df_list = [pd.read_csv(f) for f in files]
df_all = pd.concat(df_list, ignore_index=True)
print(df_all.shape)

(3846144, 119)


## Step 2. json(클러스터·구속 정보) 불러오기

`data/processed/data_description.json`은 {선수명: {연도: {average_velocity, cluster}}} 형식이다.
이후 병합을 쉽게 하기 위해 데이터프레임으로 변환한다.

In [2]:
import json

with open('../data/processed/data_description.json', 'r', encoding='utf-8') as f:
    cluster_json = json.load(f)

records = []
for player, years in cluster_json.items():
    for year, info in years.items():
        records.append({
            'player_name_json': player,
            'game_year': int(year),
            'average_velocity': info['average_velocity'],
            'cluster': info['cluster']
        })
df_json = pd.DataFrame(records)
print(df_json.shape)
print(df_json.head(10))

(3, 4)
   player_name_json  game_year  average_velocity  cluster
0     Shohei Ohtani       2023             100.0       10
1     Shohei Ohtani       2024              99.9        6
2  Justin Verlander       2024              94.5        4


## Step 3. arm_angle 컬럼 확인

Statcast 변수설명서(2021년 파일 기준)에는 arm_angle이 "값 존재: 0"으로 기록되어 있었다.
연도별로 실제 값이 채워져 있는지 확인이 필요하다.

In [3]:
arm_check = df_all.groupby('game_year')['arm_angle'].apply(lambda x: x.notna().sum())
total_check = df_all.groupby('game_year')['arm_angle'].size()
print(pd.DataFrame({'전체': total_check, 'arm_angle_존재': arm_check}))

               전체  arm_angle_존재
game_year                      
2021       765733        697923
2022       775330        715496
2023       774038        722962
2024       760248        718992
2025       770795        722894


## Step 4. 투수-시즌별 평균 프로필 집계 + json 병합

투수 이름 표기가 다르다: statcast는 "성, 이름"(예: Ohtani, Shohei), 
json은 "이름 성"(예: Shohei Ohtani). 병합 전에 이름 형식을 통일한다.

이후 투수-시즌 단위로 release_pos_x, release_pos_z, release_extension, arm_angle의 
평균을 구하고, json의 average_velocity·cluster와 병합한다.

In [4]:
def convert_name(name):
    if ',' not in name:
        return name
    last, first = [s.strip() for s in name.split(',', 1)]
    return f"{first} {last}"

df_all['player_name_std'] = df_all['player_name'].apply(convert_name)

agg_cols = ['release_pos_x', 'release_pos_z', 'release_extension', 'arm_angle']
df_agg = df_all.groupby(['player_name_std', 'game_year'])[agg_cols].mean().reset_index()

df_profile = df_json.merge(
    df_agg,
    left_on=['player_name_json', 'game_year'],
    right_on=['player_name_std', 'game_year'],
    how='left'
)
print(df_profile.shape)
print(df_profile)

C:\Users\sua08\AppData\Local\Temp\ipykernel_22600\1733660045.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_all['player_name_std'] = df_all['player_name'].apply(convert_name)


(3, 9)
   player_name_json  game_year  average_velocity  cluster   player_name_std  \
0     Shohei Ohtani       2023             100.0       10     Shohei Ohtani   
1     Shohei Ohtani       2024              99.9        6               NaN   
2  Justin Verlander       2024              94.5        4  Justin Verlander   

   release_pos_x  release_pos_z  release_extension  arm_angle  
0      -2.233195       5.692918           6.865759  35.618529  
1            NaN            NaN                NaN        NaN  
2      -1.544082       7.096684           5.950570  54.680696  


In [5]:
# 오타니 2024년 데이터가 statcast에 있는지 확인
print(df_all[(df_all['player_name_std'].str.contains('Ohtani', na=False))]['game_year'].value_counts())
print(df_all[(df_all['player_name_std'].str.contains('Ohtani', na=False))]['player_name_std'].unique())

game_year
2022    2684
2021    2127
2023    2097
2025    1089
Name: count, dtype: int64
<ArrowStringArray>
['Shohei Ohtani']
Length: 1, dtype: str


## Step 5. 유사 투수 매칭 함수

주의: 현재 `data_description.json`은 팀에서 형식 확인용으로 제공한 예시(더미) 데이터다.
아래 함수는 이 예시로 구조를 검증하며, 실제 전체 데이터로 교체되면 그대로 재사용 가능하다.

In [7]:
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import euclidean

profile_features = ['release_pos_x', 'release_pos_z', 'release_extension', 'arm_angle']

# 결측 있는 행(예: 오타니 2024)은 매칭 후보에서 제외
df_profile_valid = df_profile.dropna(subset=profile_features).copy()

scaler_profile = StandardScaler()
df_profile_valid[[c + '_z' for c in profile_features]] = scaler_profile.fit_transform(df_profile_valid[profile_features])

def find_nearest_pitcher(player_name, year, velocity_tol=2.0, fip_col=None, top_n=1):
    target_row = df_profile_valid[(df_profile_valid['player_name_json'] == player_name) & 
                                    (df_profile_valid['game_year'] == year)]
    if target_row.empty:
        return f"{player_name} {year}: 릴리스 위치/익스텐션/팔각도 데이터 없음 (매칭 불가)"
    target = target_row.iloc[0]

    candidates = df_profile_valid[
        (df_profile_valid['cluster'] == target['cluster']) &
        (df_profile_valid['average_velocity'].between(
            target['average_velocity'] - velocity_tol,
            target['average_velocity'] + velocity_tol)) &
        ~((df_profile_valid['player_name_json'] == player_name) & (df_profile_valid['game_year'] == year))
    ].copy()

    if fip_col is not None and fip_col in df_profile_valid.columns:
        candidates = candidates[candidates[fip_col] < target[fip_col]]

    if candidates.empty:
        return f"{player_name} {year}와(과) 매칭되는 후보 없음 (조건에 맞는 투수-시즌 없음)"

    z_cols = [c + '_z' for c in profile_features]
    target_vec = target[z_cols].values.astype(float)
    candidates['distance'] = candidates[z_cols].apply(
        lambda row: euclidean(row.values.astype(float), target_vec), axis=1)

    return candidates.sort_values('distance').head(top_n)[
        ['player_name_json', 'game_year', 'cluster', 'average_velocity', 'distance']]

# 테스트
print(find_nearest_pitcher("Shohei Ohtani", 2023))
print(find_nearest_pitcher("Justin Verlander", 2024))
print(find_nearest_pitcher("Shohei Ohtani", 2024))  # 데이터 없는 케이스 테스트

Shohei Ohtani 2023와(과) 매칭되는 후보 없음 (조건에 맞는 투수-시즌 없음)
Justin Verlander 2024와(과) 매칭되는 후보 없음 (조건에 맞는 투수-시즌 없음)
Shohei Ohtani 2024: 릴리스 위치/익스텐션/팔각도 데이터 없음 (매칭 불가)


## 현재 상태 및 다음 단계

- `data_description.json`은 현재 예시(더미) 데이터 3건뿐이라, 실제 매칭 결과 검증은 제한적이다.
- 함수 로직(같은 클러스터 + 구속 유사 + [FIP 미제공 시 생략] 필터링 → 표준화한 
  release_pos/extension/arm_angle 기준 유클리드 거리 최솟값)은 정상 작동 확인됨 
  (매칭 후보가 없을 때 에러 없이 "없음" 메시지 반환).
- 팀에서 실제 전체 투수-시즌 json으로 교체되면, `data/processed/data_description.json` 경로만 
  유지된다면 코드 수정 없이 재실행 가능.
- FIP 데이터가 별도로 제공되면 `df_profile`에 merge 후 `find_nearest_pitcher(..., fip_col='FIP')`로 
  호출하면 자동으로 필터 추가됨.

---

## Ver2. 마할라노비스 거리(Mahalanobis Distance) 기반 매칭

Ver1(유클리드 거리)은 4개 변수(release_pos_x, release_pos_z, release_extension, arm_angle)를
서로 독립적이라고 가정하고 거리를 계산한다. 하지만 실제로는 release_pos_x와 release_pos_z처럼
변수 간 상관관계가 존재할 수 있다.

마할라노비스 거리는 변수 간 공분산 구조를 반영해 이 상관관계를 보정한 "진짜 의미 있는 거리"를 계산한다.
연구계획서 10번 섹션에도 "Mahalanobis distance를 baseline으로 사용하되 공분산 추정 안정성을 점검한다"고
명시되어 있다.

**주의**: 현재 예시 데이터는 유효 표본이 2건뿐이라(오타니 2023, 벌랜더 2024) 공분산 행렬 추정이
불안정(특이행렬에 가까움)할 수 있다. 실제 데이터로 교체되면 표본이 충분해져 안정적으로 계산된다.
여기서는 pseudo-inverse(`np.linalg.pinv`)로 특이행렬 문제를 임시 방지한다.

In [ ]:
from scipy.spatial.distance import mahalanobis
import numpy as np

def find_nearest_pitcher_v2(player_name, year, velocity_tol=2.0, fip_col=None, top_n=1):
    target_row = df_profile_valid[(df_profile_valid['player_name_json'] == player_name) &
                                    (df_profile_valid['game_year'] == year)]
    if target_row.empty:
        return f"{player_name} {year}: 릴리스 위치/익스텐션/팔각도 데이터 없음 (매칭 불가)"
    target = target_row.iloc[0]

    candidates = df_profile_valid[
        (df_profile_valid['cluster'] == target['cluster']) &
        (df_profile_valid['average_velocity'].between(
            target['average_velocity'] - velocity_tol,
            target['average_velocity'] + velocity_tol)) &
        ~((df_profile_valid['player_name_json'] == player_name) & (df_profile_valid['game_year'] == year))
    ].copy()

    if fip_col is not None and fip_col in df_profile_valid.columns:
        candidates = candidates[candidates[fip_col] < target[fip_col]]

    if candidates.empty:
        return f"{player_name} {year}와(과) 매칭되는 후보 없음 (조건에 맞는 투수-시즌 없음)"

    z_cols = [c + '_z' for c in profile_features]

    cov = np.cov(df_profile_valid[z_cols].values, rowvar=False)
    inv_cov = np.linalg.pinv(cov)

    target_vec = target[z_cols].values.astype(float)
    candidates['distance'] = candidates[z_cols].apply(
        lambda row: mahalanobis(row.values.astype(float), target_vec, inv_cov), axis=1)

    return candidates.sort_values('distance').head(top_n)[
        ['player_name_json', 'game_year', 'cluster', 'average_velocity', 'distance']]

print(find_nearest_pitcher_v2("Shohei Ohtani", 2023))
print(find_nearest_pitcher_v2("Justin Verlander", 2024))